# Membership v3 파생변수 생성

---

## 개요
- **입력**: promotion_0_membership_v2.csv (원본 15개 컬럼)
- **출력**: promotion_0_membership_v3.csv (원본 15 + 파생 53 = **68개 컬럼**)

## 파생변수 구성 (53개)
| 구분 | 개수 | 주요 변수 |
|------|------|----------|
| 날짜/시간 | 2 | duration_days, reg_hour_group |
| 가격 | 3 | is_usd, price_per_day, price_per_screen |
| 상품/기기 | 4 | is_new_product, is_family_plan, device_group, is_apple_ecosystem |
| 요금제 | 3 | is_basic, is_standard, is_premium |
| 인구통계 | 8 | age_group, gender_enc, age_x_screen, verified_x_age 등 |
| 장르 | 11 | genre_diversity, korean_ratio, avg_showtime, genre_*_ratio x 8 |
| View History | 24 | completion_rate, recency, retention_w2/w3_ratio 등 |
| 교호작용 | 2 | stream_watch_interaction, plan_promotion |

In [20]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

BASE     = Path('..').resolve()
SPLIT    = BASE / 'data/02_interim/260504_promotion_split'
DATA_OUT = BASE / 'data/02_interim/260506_feature_engineering'
DATA_OUT.mkdir(exist_ok=True)
print('설정 완료')

설정 완료


In [21]:
# 데이터 로드
df = pd.read_csv(SPLIT / 'promotion_0_membership_v2.csv', encoding='utf-8-sig')

um = pd.concat([
    pd.read_csv(SPLIT / 'promotion_0_user_mapping_v2.csv', encoding='utf-8-sig'),
    pd.read_csv(SPLIT / 'promotion_1_user_mapping_v2.csv', encoding='utf-8-sig'),
], ignore_index=True).drop_duplicates()

vh = pd.read_csv(SPLIT / 'promotion_0_view_history_v2.csv', encoding='utf-8-sig').drop_duplicates()

movie = pd.read_csv(BASE / 'data/02_interim/Movie/movie_5171.csv', encoding='utf-8-sig')

print(f'membership:   {df.shape}')
print(f'view_history: {vh.shape}')
print(f'movie:        {movie.shape}')

membership:   (7323, 15)
view_history: (50858, 5)
movie:        (5171, 5)


## 1. 날짜/시간 기반 파생변수

In [22]:
df['reg_date'] = pd.to_datetime(df['reg_date'])
df['end_date'] = pd.to_datetime(df['end_date'])

df['duration_days']  = (df['end_date'] - df['reg_date']).dt.days
df['reg_hour_group'] = pd.cut(
    df['reg_hour'], bins=[-1, 5, 11, 17, 23], labels=[0, 1, 2, 3]
).astype(float).astype('Int64')

print('날짜/시간 파생변수 완료 (2개): duration_days, reg_hour_group')
df[['duration_days', 'reg_hour_group']].describe()

날짜/시간 파생변수 완료 (2개): duration_days, reg_hour_group


,duration_days,reg_hour_group
count,7323.000000,7323.0
mean,30.057490,1.735901
std,6.146184,1.117101
min,0.000000,0.0
25%,31.000000,1.0
50%,31.000000,2.0
75%,32.000000,3.0
max,32.000000,3.0


df['reg_date'] = pd.to_datetime(df['reg_date'])
df['end_date'] = pd.to_datetime(df['end_date'])

df['duration_days']  = (df['end_date'] - df['reg_date']).dt.days
df['reg_hour_group'] = pd.cut(
    df['reg_hour'], bins=[-1, 5, 11, 17, 23], labels=[0, 1, 2, 3]
).astype(float).astype('Int64')

print('✅ 날짜/시간 파생변수 완료 (2개)')
df[['duration_days', 'reg_hour_group']].describe()

In [23]:
df['is_usd']          = (df['price'] < 100).astype(int)
df['price_per_day']   = (df['price'] / df['duration_days'].replace(0, np.nan)).round(2)
df['price_per_screen']= (df['price'] / df['max_screen']).round(2)

print('가격 파생변수 완료 (3개): is_usd, price_per_day, price_per_screen')
print('is_usd 분포:', df['is_usd'].value_counts().to_dict())

가격 파생변수 완료 (3개): is_usd, price_per_day, price_per_screen
is_usd 분포: {0: 4843, 1: 2480}


df['is_usd']           = (df['price'] < 100).astype(int)
df['price_per_day']    = (df['price'] / df['duration_days'].replace(0, __import__('numpy').nan)).round(2)
df['price_per_screen'] = (df['price'] / df['max_screen']).round(2)

print('✅ 가격 파생변수 완료 (3개)')
print('is_usd 분포:', df['is_usd'].value_counts().to_dict())

In [24]:
df['is_new_product'] = (
    df['product_code'].str.extract(r'pk_(\d+)').astype(float) >= 2000
).astype(int).values.flatten()

df['is_family_plan'] = (df['max_screen'] > 1).astype(int)

df['device_group'] = df['payment_device'].map({
    'ios': 'mobile', 'android': 'mobile', 'mobile': 'mobile',
    'pc': 'pc', 'smarttv': 'tv', 'ott': 'tv',
})

df['is_apple_ecosystem'] = (
    (df['payment_device'] == 'ios') & (df['billing_method'] == 151)
).astype(int)

print('상품/기기 파생변수 완료 (4개)')
print('device_group:', df['device_group'].value_counts().to_dict())

상품/기기 파생변수 완료 (4개)
device_group: {'mobile': 6118, 'pc': 985, 'tv': 220}


df['is_new_product'] = (
    df['product_code'].str.extract(r'pk_(\d+)').astype(float) >= 2000
).astype(int).values.flatten()

df['is_family_plan'] = (df['max_screen'] > 1).astype(int)

df['device_group'] = df['payment_device'].map({
    'ios': 'mobile', 'android': 'mobile', 'mobile': 'mobile',
    'pc': 'pc', 'smarttv': 'tv', 'ott': 'tv',
})

# iOS 기기 + Apple Pay -> 애플 생태계 유저
df['is_apple_ecosystem'] = (
    (df['payment_device'] == 'ios') & (df['billing_method'] == 151)
).astype(int)

print('✅ 상품/기기 파생변수 완료 (4개)')
print('device_group:', df['device_group'].value_counts().to_dict())
print('is_apple_ecosystem:', df['is_apple_ecosystem'].sum(), '명')

In [25]:
df['age_group'] = pd.cut(
    df['age'], bins=[0, 19, 29, 39, 49, 120],
    labels=[0, 1, 2, 3, 4]  # 10/20/30/40/50대+
).astype(float).astype('Int64')

df['gender_enc']           = df['gender'].map({'M': 1, 'F': 0}).fillna(2).astype(int)
df['age_x_screen']         = df['age'] * df['max_screen']
df['is_senior_unverified'] = ((df['age'] >= 50) & (df['is_user_verified'] == 0)).astype(int)
df['is_young_unverified']  = ((df['age'] <= 25) & (df['is_user_verified'] == 0)).astype(int)

print('인구통계 파생변수 완료')
print('age_group 분포:', df['age_group'].value_counts().sort_index().to_dict())

인구통계 파생변수 완료
age_group 분포: {0: 76, 1: 1549, 2: 1513, 3: 3764, 4: 421}


## 5. 장르 기반 파생변수 (View History + Movie)

## 5-1. 요금제 Tier (max_screen 기반)

In [26]:
# 요금제 Tier (max_screen 기반)
df['is_basic']   = (df['max_screen'] == 1).astype(int)  # 1화면
df['is_standard']= (df['max_screen'] == 2).astype(int)  # 2화면
df['is_premium'] = (df['max_screen'] == 4).astype(int)  # 4화면

print('✅ 요금제 tier 완료')
print(df[['is_basic','is_standard','is_premium']].sum())

✅ 요금제 tier 완료
is_basic       4781
is_standard    1822
is_premium      720
dtype: int64


In [27]:
# showTM → 분 변환
def parse_showtime(s):
    if pd.isna(s): return np.nan
    h = re.search(r'(\d+)시간', str(s))
    m = re.search(r'(\d+)분',   str(s))
    hours, mins = (int(h.group(1)) if h else 0), (int(m.group(1)) if m else 0)
    total = hours * 60 + mins
    return total if total > 0 else np.nan

movie['showtime_min'] = movie['showTM'].apply(parse_showtime)

# VIEW HISTORY에 USER_KEY + 영화 정보 붙이기
vh_key   = vh.merge(um, on='USER_NUM', how='left')
vh_movie = vh_key.merge(
    movie[['MOVIE_ID', 'genre', 'country', 'showtime_min']],
    left_on='MOVIE_NUM', right_on='MOVIE_ID', how='left'
)
print(f'시청이력+영화: {vh_movie.shape}, 장르 결측: {vh_movie["genre"].isna().mean()*100:.1f}%')

시청이력+영화: (50858, 10), 장르 결측: 1.8%


In [28]:
TARGET_GENRES = ['액션', '드라마', '로맨스', '스릴러', '애니메이션', '공포', '코미디', 'SF']

rows = []
for user_key, grp in vh_movie.groupby('USER_KEY'):
    row = {'USER_KEY': user_key}

    # 장르 다양성
    all_genres = set()
    for g in grp['genre'].dropna():
        all_genres.update(x.strip() for x in g.split(','))
    row['genre_diversity'] = len(all_genres)

    # 한국 영화 비율
    total = grp['country'].notna().sum()
    row['korean_ratio'] = grp['country'].str.contains('한국', na=False).sum() / total if total else 0

    # 평균 러닝타임
    row['avg_showtime'] = grp['showtime_min'].mean()

    # 장르별 비율
    genre_total = grp['genre'].notna().sum()
    for g in TARGET_GENRES:
        row[f'genre_{g}_ratio'] = (
            grp['genre'].dropna().str.contains(g).sum() / genre_total if genre_total else 0
        )
    rows.append(row)

genre_df = pd.DataFrame(rows)
print(f'장르 파생변수 생성 완료: {genre_df.shape}')
genre_df.head(3)

장르 파생변수 생성 완료: (7140, 12)


,USER_KEY,genre_diversity,korean_ratio,avg_showtime,genre_액션_ratio,genre_드라마_ratio,genre_로맨스_ratio,genre_스릴러_ratio,genre_애니메이션_ratio,genre_공포_ratio,genre_코미디_ratio,genre_SF_ratio
0,0006075c3c18078eb09940cd27c6359a96a2a17fce8055...,7,0.0,87.000000,0.200000,0.6,0.000000,0.000000,0.6,0.000000,0.400000,0.000000
1,0019ebcf13ea62a20b0e6626103f4d2164e61c64355b9d...,9,0.0,111.500000,0.166667,0.5,0.666667,0.333333,0.0,0.000000,0.666667,0.166667
2,001e7cb9cd0658e839caf6f36441c895f4dede5c53a4a8...,9,0.0,120.636364,0.222222,1.0,0.222222,0.444444,0.0,0.111111,0.111111,0.000000


## 6. 합치고 저장

## 6. View History 기반 파생변수

In [29]:
# vh_movie는 cell-13에서 이미 생성됨 (genre + showtime_min 포함)
# USER_KEY 중복 제거 후 매핑 (동일 유저 다중 구독 있을 수 있음)
df_dedup = df.drop_duplicates(subset='USER_KEY', keep='first')
reg_map  = df_dedup.set_index('USER_KEY')['reg_date']
end_map  = df_dedup.set_index('USER_KEY')['end_date']

vh_movie['reg_date_ref'] = pd.to_datetime(vh_movie['USER_KEY'].map(reg_map))
vh_movie['end_date_ref'] = pd.to_datetime(vh_movie['USER_KEY'].map(end_map))
vh_movie['watch_date']   = pd.to_datetime(vh_movie['watch_day'].astype(str), format='%Y%m%d', errors='coerce')
vh_movie['days_since_reg'] = (vh_movie['watch_date'] - vh_movie['reg_date_ref']).dt.days
vh_movie['is_weekend']     = vh_movie['watch_date'].dt.dayofweek.isin([5, 6])
vh_movie['obs_week']       = pd.cut(vh_movie['days_since_reg'], bins=[-1,6,13,200], labels=[1,2,3])

# 신작 여부 (ott_release_month가 202103인 영화)
new_movies_2103 = set(
    pd.read_csv('../data/02_interim/260504_promotion_split/promotion_0_movie_master_v2.csv',
                encoding='utf-8-sig')
    .query('ott_release_month == 202103')['MOVIE_NUM']
)
vh_movie['is_new'] = vh_movie['MOVIE_NUM'].isin(new_movies_2103)

rows_vh = []
for user_key, grp in vh_movie.groupby('USER_KEY'):
    row = {'USER_KEY': user_key}

    row['total_watch_time'] = grp['watch_time(min)'].sum()
    row['total_sessions']   = len(grp)
    row['unique_movies']    = grp['MOVIE_NUM'].nunique()
    row['active_days']      = grp['watch_date'].nunique()
    row['avg_session_time'] = grp['watch_time(min)'].mean()
    row['has_watch_history']= 1

    dur_days = max((pd.to_datetime(end_map.get(user_key, pd.NaT)) -
                    pd.to_datetime(reg_map.get(user_key, pd.NaT))).days, 1)
    row['activity_rate'] = round(row['active_days'] / dur_days, 4)
    row['watch_per_day'] = round(row['total_sessions'] / row['active_days'], 4) if row['active_days'] > 0 else 0

    row['avg_rewatch_ratio'] = round(
        max(row['total_sessions'] - row['unique_movies'], 0) / row['total_sessions'], 4
    )

    row['signup_to_first_watch'] = grp['days_since_reg'].min()
    last_watch_date = grp['watch_date'].max()
    end_date_dt     = pd.to_datetime(end_map.get(user_key, pd.NaT))
    row['recency']  = (end_date_dt - last_watch_date).days if pd.notna(last_watch_date) else dur_days

    valid = grp[grp['showtime_min'].notna() & (grp['showtime_min'] > 0)]
    row['completion_rate'] = round(
        (valid['watch_time(min)'] / valid['showtime_min']).clip(0, 1).mean(), 4
    ) if len(valid) > 0 else 0

    row['weekend_watch_ratio'] = round(grp['is_weekend'].mean(), 4)
    row['weekday_watch_ratio'] = round(1 - row['weekend_watch_ratio'], 4)

    sorted_days = sorted(grp['watch_date'].dropna().unique())
    row['max_gap_between_watch_days'] = max(
        [(sorted_days[i+1]-sorted_days[i]).days for i in range(len(sorted_days)-1)], default=0
    )

    for w in [1, 2, 3]:
        wg = grp[grp['obs_week'] == str(w)]
        row[f'dur_w{w}'] = wg['watch_time(min)'].sum()

    row['retention_w2']       = int(row['dur_w2'] > 0)
    row['retention_w3']       = int(row['dur_w3'] > 0)
    row['retention_w2_ratio'] = round(row['dur_w2'] / (row['dur_w1'] + 1e-6), 4)
    row['retention_w3_ratio'] = round(row['dur_w3'] / (row['dur_w2'] + 1e-6), 4)

    row['is_new_movie'] = round(grp['is_new'].mean(), 4)

    daily_sessions = grp.groupby('watch_date').size()
    row['binge_day_count'] = int((daily_sessions >= 3).sum())

    rows_vh.append(row)

vh_feat_df = pd.DataFrame(rows_vh)
print(f'View History 파생변수 생성 완료: {vh_feat_df.shape}')
vh_feat_df.head(3)

View History 파생변수 생성 완료: (7140, 25)


,USER_KEY,total_watch_time,total_sessions,unique_movies,active_days,avg_session_time,has_watch_history,activity_rate,watch_per_day,avg_rewatch_ratio,...,max_gap_between_watch_days,dur_w1,dur_w2,dur_w3,retention_w2,retention_w3,retention_w2_ratio,retention_w3_ratio,is_new_movie,binge_day_count
0,0006075c3c18078eb09940cd27c6359a96a2a17fce8055...,452,5,4,3,90.400000,1,0.0968,1.6667,0.2000,...,10,0,0,0,0,0,0.0,0.0,0.0,1
1,0019ebcf13ea62a20b0e6626103f4d2164e61c64355b9d...,311,6,6,3,51.833333,1,0.0968,2.0000,0.0000,...,11,0,0,0,0,0,0.0,0.0,0.0,1
2,001e7cb9cd0658e839caf6f36441c895f4dede5c53a4a8...,858,11,7,6,78.000000,1,0.1935,1.8333,0.3636,...,6,0,0,0,0,0,0.0,0.0,0.0,1


In [32]:
result = df.merge(genre_df,    on='USER_KEY', how='left')
result = result.merge(vh_feat_df, on='USER_KEY', how='left')

genre_cols = (
    ['genre_diversity', 'korean_ratio', 'avg_showtime'] +
    [f'genre_{g}_ratio' for g in TARGET_GENRES]
)

vh_cols = [
    'total_watch_time','total_sessions','unique_movies','active_days',
    'avg_session_time','activity_rate','watch_per_day','avg_rewatch_ratio',
    'signup_to_first_watch','recency','completion_rate',
    'weekend_watch_ratio','weekday_watch_ratio','max_gap_between_watch_days',
    'dur_w1','dur_w2','dur_w3',
    'retention_w2','retention_w3','retention_w2_ratio','retention_w3_ratio',
    'is_new_movie','binge_day_count','has_watch_history',
]
result[genre_cols + vh_cols] = result[genre_cols + vh_cols].fillna(0)

result['stream_watch_interaction'] = result['max_screen'] * result['total_watch_time']
result['plan_promotion']           = result['max_screen'] * result['is_promotion']

out_path = DATA_OUT / 'promotion_0_membership_v3.csv'
result.to_csv(out_path, index=False, encoding='utf-8-sig')

new_features = (
    ['duration_days','reg_hour_group',
     'is_usd','price_per_day','price_per_screen',
     'is_new_product','is_family_plan','device_group','is_apple_ecosystem',
     'is_basic','is_standard','is_premium',
     'age_group','gender_enc','age_x_screen','verified_x_age',
     'is_senior_unverified','is_young_unverified','is_long_sub','hour_x_weekend']
    + genre_cols + vh_cols
    + ['stream_watch_interaction','plan_promotion']
)

print(f'저장 완료: {out_path}')
print(f'원본: 15개  |  파생변수: {len(new_features)}개  |  최종: {result.shape[1]}개 컬럼')
print(f'최종 shape: {result.shape}')
result.head(3)
